# Starter Notebook: Qwen 2B LoRA for Text-to-SVG (Kaggle)

This starter is built from the resources in `contest_docs`:
- Data resources: `contest_docs/03_Data_Design.md`
- Baseline and starter guidance: `contest_docs/05_Baselines_and_Starter_Notebooks.md`
- Kaggle implementation notes: `contest_docs/06_Kaggle_Implementation_Guide.md`

Goal: provide a practical scaffold for Qwen-2B-class fine-tuning + submission generation.

## Referenced Data and Docs

### Dataset resources from `contest_docs/03_Data_Design.md`
- `OmniSVG/MMSVG-Icon`
- `xingxm/SVGX-Core-250k`
- `xingxm/SVGX-SFT-1M` (recommended subset: `SVGX_SFT_GEN_51k.json`)
- `nyuuzyou/svgfind`
- `starvector/svg-icons`
- `thesantatitan/deepseek-svg-dataset`
- `InternSVG/SArena` (evaluation benchmark)

### Qwen 2B fine-tuning references from `contest_docs/05` and `contest_docs/06`
- Unsloth Qwen fine-tune docs: https://unsloth.ai/docs/models/qwen3.5/fine-tune
- Qwen3.5-2B Vision notebook: https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(2B)_Vision.ipynb

Note: this notebook is written as a reusable starter. You may need to adjust exact model IDs and column names to match the latest upstream datasets.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
cd drive/MyDrive/NYU

/content/drive/MyDrive/NYU


In [3]:
cd ECE-GY 7123 DeepLearning

/content/drive/MyDrive/NYU/ECE-GY 7123 DeepLearning


In [4]:
cd Midterm/test

/content/drive/MyDrive/NYU/ECE-GY 7123 DeepLearning/Midterm/test


In [5]:
pip install cairosvg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 5.1 MB/s eta 0:00:00


In [6]:
pip install torch

In [7]:
import torch
print(torch.__version__)
print(torch.bfloat16)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.10.0+cu128
torch.bfloat16
True
NVIDIA RTX PRO 6000 Blackwell Server Edition


In [8]:
pip install transformers

In [9]:
pip install peft

In [10]:
pip install -U bitsandbytes>=0.46.1

In [11]:
# Uncomment in a fresh Kaggle notebook environment.
%pip install -q unsloth datasets trl transformers accelerate peft bitsandbytes pandas lxml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.9/62.9 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.8/403.8 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 129.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 30.6 MB/s eta 0:00:00


In [12]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch

from datasets import concatenate_datasets, load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Torch: 2.10.0+cu128
CUDA available: True


In [13]:
# Core training config.
# Keep runtime targets in line with contest_docs guidance (roughly <= 6-8 hours training).
"""
CONFIG = {
    "model_name": "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",  # Verify exact ID from the linked Unsloth notebook.
    "max_seq_length": 2048,
    "lora_r": 16,
    "lora_alpha": 16,
    "learning_rate": 2e-4,
    "num_train_epochs": 1,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "logging_steps": 20,
    "eval_steps": 100,
    "save_steps": 200,
    "max_train_samples_per_source": 12000,
    "eval_size": 0.02,
    "output_dir": "/kaggle/working/qwen2b_svg_lora",
}
"""
CONFIG = {
    "model_name": "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    #"model_name": "unsloth/Qwen2.5-4B-Instruct-bnb-4bit",
    "max_seq_length": 2048,
    "lora_r": 8,
    "lora_alpha": 16,
    "learning_rate": 1e-4,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 4,
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "logging_steps": 20,
    "eval_steps": 100,
    "save_steps": 200,
    "max_train_samples_per_source": 50000,
    "eval_size": 0.5,
    "output_dir": "/content/qwen2b_svg_lora",
}
CONFIG

{'model_name': 'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit',
 'max_seq_length': 2048,
 'lora_r': 8,
 'lora_alpha': 16,
 'learning_rate': 0.0001,
 'num_train_epochs': 3,
 'per_device_train_batch_size': 1,
 'gradient_accumulation_steps': 4,
 'warmup_ratio': 1,
 'weight_decay': 0.01,
 'logging_steps': 20,
 'eval_steps': 100,
 'save_steps': 200,
 'max_train_samples_per_source': 50000,
 'eval_size': 2,
 'output_dir': '/content/qwen2b_svg_lora'}

In [14]:
# Data catalog using the resources listed in contest_docs/03_Data_Design.md.
"""
DATASET_CATALOG = {
    "OmniSVG/MMSVG-Icon": {
        "split": "train",
        "prompt_fields": ["description", "keywords", "detail", "prompt", "text"],
        "svg_fields": ["svg", "picosvg", "completion", "target"],
    },

    "xingxm/SVGX-Core-250k": {
        "split": "train",
        "prompt_fields": ["qwen_caption", "img_analysis", "blip_caption", "name"],
        "svg_fields": ["svg_code"],
    },

    "xingxm/SVGX-SFT-1M": {
        "split": "train",
        "prompt_fields": ["prompt", "instruction", "input", "query"],
        "svg_fields": ["completion", "output", "svg", "response"],
    },
    "thesantatitan/deepseek-svg-dataset": {
        "split": "train",
        "prompt_fields": ["prompt", "instruction", "input"],
        "svg_fields": ["completion", "output", "svg"],
    },

    "local_train_csv": {
        "split": "train",
        "prompt_fields": ["prompt"],
        "svg_fields": ["svg"],
    },
}
"""
DATASET_CATALOG = {
    "local_train_csv": {
        "split": "train",
        "prompt_fields": ["prompt"],
        "svg_fields": ["svg"],
    }
}

ACTIVE_SOURCES = [
    "local_train_csv",
]

print(DATASET_CATALOG)
print(ACTIVE_SOURCES)


{'local_train_csv': {'split': 'train', 'prompt_fields': ['prompt'], 'svg_fields': ['svg']}}
['local_train_csv']


In [15]:
def _pick_first_non_empty(example, keys):
    for key in keys:
        if key in example and example[key] is not None:
            val = str(example[key]).strip()
            if val:
                return val
    return ""


def to_prompt_svg(example, prompt_fields, svg_fields):
    prompt = _pick_first_non_empty(example, prompt_fields)
    svg = _pick_first_non_empty(example, svg_fields)
    if not svg.lower().startswith("<svg"):
        return {"prompt": "", "svg": ""}
    return {"prompt": prompt, "svg": svg}

"""
def load_source_dataset(dataset_id, cfg, max_samples):
    print(f"Loading {dataset_id} ...")
    ds = load_dataset(dataset_id, split=cfg["split"])
    if max_samples and len(ds) > max_samples:
        ds = ds.shuffle(seed=SEED).select(range(max_samples))
    ds = ds.map(
        lambda ex: to_prompt_svg(ex, cfg["prompt_fields"], cfg["svg_fields"]),
        remove_columns=ds.column_names,
        desc=f"normalizing {dataset_id}",
    )
    ds = ds.filter(lambda x: bool(x["prompt"]) and bool(x["svg"]))
    print(f"{dataset_id}: {len(ds)} usable rows")
    return ds
"""
def load_source_dataset(dataset_id, cfg, max_samples):
    print(f"Loading {dataset_id} ...")

    if dataset_id == "local_train_csv":
        ds = load_dataset(
            "csv",
            data_files="train.csv",   # 如果不在同目錄，改成你的實際路徑
            split="train",
        )
    else:
        ds = load_dataset(dataset_id, split=cfg["split"])

    if max_samples and len(ds) > max_samples:
        ds = ds.shuffle(seed=SEED).select(range(max_samples))

    ds = ds.map(
        lambda ex: to_prompt_svg(ex, cfg["prompt_fields"], cfg["svg_fields"]),
        remove_columns=ds.column_names,
        desc=f"normalizing {dataset_id}",
    )

    ds = ds.filter(lambda x: bool(x["prompt"]) and bool(x["svg"]))
    print(f"{dataset_id}: {len(ds)} usable rows")
    return ds

In [16]:
"""
datasets_ok = []
for source in ACTIVE_SOURCES:
    try:
        ds = load_source_dataset(
            source,
            DATASET_CATALOG[source],
            CONFIG["max_train_samples_per_source"],
        )
        datasets_ok.append(ds)
    except Exception as e:
        print(f"Skipping {source}: {type(e).__name__}: {e}")

if not datasets_ok:
    raise RuntimeError("No dataset loaded. Check dataset IDs, internet access, and schema fields.")

train_raw = datasets_ok[0] if len(datasets_ok) == 1 else concatenate_datasets(datasets_ok)
train_raw = train_raw.shuffle(seed=SEED)

splits = train_raw.train_test_split(test_size=CONFIG["eval_size"], seed=SEED)
train_ds = splits["train"]
eval_ds = splits["test"]

print(f"Train rows: {len(train_ds)}")
print(f"Eval rows: {len(eval_ds)}")
train_ds[0]
"""
datasets_ok = []
for source in ACTIVE_SOURCES:
    try:
        ds = load_source_dataset(
            source,
            DATASET_CATALOG[source],
            CONFIG["max_train_samples_per_source"],
        )
        datasets_ok.append(ds)
    except Exception as e:
        print(f"Skipping {source}: {type(e).__name__}: {e}")

if not datasets_ok:
    raise RuntimeError("No dataset loaded. Check dataset IDs, internet access, and schema fields.")

train_raw = datasets_ok[0] if len(datasets_ok) == 1 else concatenate_datasets(datasets_ok)
train_raw = train_raw.shuffle(seed=SEED)

def keep_short_svg(x):
    svg = x["svg"]
    return (
        len(svg) <= 1600 and
        svg.count("<path") <= 10 and
        svg.count("<circle") <= 8 and
        svg.count("<rect") <= 8
    )

train_raw = train_raw.filter(keep_short_svg)

splits = train_raw.train_test_split(test_size=CONFIG["eval_size"], seed=SEED)
train_ds = splits["train"]
eval_ds = splits["test"]

print(f"Train rows: {len(train_ds)}")
print(f"Eval rows: {len(eval_ds)}")
print(train_ds[0])

Loading local_train_csv ...


Generating train split: 0 examples [00:00, ? examples/s]

normalizing local_train_csv:   0%|          | 0/50000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/50000 [00:00<?, ? examples/s]

local_train_csv: 50000 usable rows


Filter:   0%|          | 0/50000 [00:00<?, ? examples/s]

Train rows: 18384
Eval rows: 2
{'prompt': 'The image features a teal-colored circular shape with an arrow pointing left, indicating a refresh or reset action.', 'svg': '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0.0 0.0 200.0 200.0" height="200.0px" width="200.0px"><path fill="#00AAC1" fill-opacity="1.0"  filling="0" d="M138.5 30.0 L175.0 30.0 C178.0 30.0 180.0 28.0 180.0 25.0 C180.0 22.0 178.0 20.0 175.0 20.0 L125.0 20.0 C124.5 20.0 120.0 20.0 120.0 25.0 L120.0 75.0 C120.0 78.0 122.0 80.0 125.0 80.0 C128.0 80.0 130.0 78.0 130.0 75.0 L130.0 37.0 C153.5 48.0 170.0 72.5 170.0 100.0 C170.0 138.5 138.5 170.0 100.0 170.0 C61.5 170.0 30.0 138.5 30.0 100.0 C30.0 64.5 56.0 35.5 90.0 31.0 L90.0 21.0 C50.5 25.5 20.0 58.999996185302734 20.0 100.0 C20.0 144.0 56.0 180.0 100.0 180.0 C144.0 180.0 180.0 144.0 180.0 100.0 C180.0 70.0 163.5 43.5 138.5 30.0 Z"></path></svg>'}


In [17]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total params: {total_params:,}")
print(f"In billions: {total_params / 1e9:.2f}B")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.18: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

unsloth/qwen2.5-1.5b-instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.3.18 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Total params: 897,848,832
In billions: 0.90B


In [18]:

SYSTEM_PROMPT = (
    #"You generate compact, valid SVG markup from user requests. "
    #"Return only SVG code with a single root <svg> element."
    "You generate compact, valid SVG markup from user requests. "
    "Return only valid SVG markup. No explanations or extra text."
)

"""
def format_sft_text(example):
    text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{example['prompt']}<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{example['svg']}<|im_end|>"
    )
    return {"text": text}


train_text = train_ds.map(format_sft_text, remove_columns=train_ds.column_names)
eval_text = eval_ds.map(format_sft_text, remove_columns=eval_ds.column_names)
"""
def format_sft_text(example):
    text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{example['prompt']}<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{example['svg']}<|im_end|>"
    )
    return {"text": text}

train_text_all = train_ds.map(format_sft_text, remove_columns=train_ds.column_names)
eval_text_all = eval_ds.map(format_sft_text, remove_columns=eval_ds.column_names)

def keep_short_tokenized(example):
    n_tokens = len(tokenizer(example["text"], add_special_tokens=False)["input_ids"])
    return n_tokens <= 1700

train_text = train_text_all.filter(keep_short_tokenized)
eval_text = eval_text_all.filter(keep_short_tokenized)

print("Filtered train rows:", len(train_text))
print("Filtered eval rows:", len(eval_text))
print(train_text[0]["text"][:400])

Map:   0%|          | 0/18384 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Filter:   0%|          | 0/18384 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2 [00:00<?, ? examples/s]

Filtered train rows: 18384
Filtered eval rows: 2
<|im_start|>system
You generate compact, valid SVG markup from user requests. Return only valid SVG markup. No explanations or extra text.<|im_end|>
<|im_start|>user
The image features a teal-colored circular shape with an arrow pointing left, indicating a refresh or reset action.<|im_end|>
<|im_start|>assistant
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0.0 0.0 200.0 200.0" height="200.0px"


In [19]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=CONFIG["logging_steps"],
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    report_to="none",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_text,
    eval_dataset=eval_text,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    packing=False,
    args=training_args,
)

train_result = trainer.train()
train_result

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=52):   0%|          | 0/18384 [00:00<?, ? examples/s]

num_proc must be <= 2. Reducing num_proc to 2 for dataset of size 2.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 18,384 | Num Epochs = 3 | Total steps = 13,788
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 9,232,384 of 1,552,946,688 (0.59% trained)


Step,Training Loss,Validation Loss
100,0.588732,0.527000
200,0.562605,0.509243
300,0.444991,0.497681
400,0.503541,0.495660
500,0.514102,0.472001
600,0.458991,0.472675
700,0.493000,0.468078
800,0.477582,0.465805
900,0.436780,0.465693
1000,0.461093,0.450888


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

TrainOutput(global_step=13788, training_loss=0.38504658606144393, metrics={'train_runtime': 4594.4496, 'train_samples_per_second': 12.004, 'train_steps_per_second': 3.001, 'total_flos': 3.1961272665136435e+17, 'train_loss': 0.38504658606144393, 'epoch': 3.0})

In [20]:
os.makedirs(CONFIG["output_dir"], exist_ok=True)
trainer.save_model(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])

print(f"Saved adapter + tokenizer to: {CONFIG['output_dir']}")
print(os.listdir(CONFIG["output_dir"]))

Saved adapter + tokenizer to: /content/qwen2b_svg_lora
['training_args.bin', 'checkpoint-13788', 'tokenizer_config.json', 'tokenizer.json', 'checkpoint-13600', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'README.md']


In [21]:
from unsloth import FastLanguageModel
import torch
import re
import xml.etree.ElementTree as ET

infer_model, infer_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)

# LoRA adapter
infer_model.load_adapter(CONFIG["output_dir"])

# Move to GPU
infer_model = infer_model.to("cuda")
infer_model.eval()

SVG_REGEX = re.compile(r"<svg\b[^>]*>[\s\S]*?</svg>", flags=re.IGNORECASE)

def extract_svg(text: str) -> str:
    matches = SVG_REGEX.findall(text)
    return matches[-1].strip() if matches else ""

def is_valid_svg(svg_text: str) -> bool:
    if not isinstance(svg_text, str) or not svg_text.strip():
        return False
    try:
        root = ET.fromstring(svg_text)
        if "}" in root.tag:
            tag = root.tag.split("}", 1)[1]
        else:
            tag = root.tag
        return tag == "svg"
    except ET.ParseError:
        return False

def fallback_svg(_prompt: str) -> str:
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="black"/>'
        '</svg>'
    )

def postprocess_svg(svg: str) -> str:
    if not isinstance(svg, str):
        return ""
    svg = re.sub(r'\bfilling\s*=\s*"([^"]*)"', r'fill-rule="\1"', svg, flags=re.IGNORECASE)
    return svg.strip()

def generate_svg_once(
    prompt: str,
    temperature: float,
    top_p: float,
    repetition_penalty: float,
    max_cap: int = 768,
    debug: bool = False,
) -> str:
    chat_text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{prompt}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

    inputs = infer_tokenizer(
        chat_text,
        return_tensors="pt",
        add_special_tokens=False,
    ).to("cuda")

    input_len = inputs["input_ids"].shape[1]
    max_new_tokens = max(128, min(max_cap, CONFIG["max_seq_length"] - input_len - 64))

    with torch.no_grad():
        output_ids = infer_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            pad_token_id=infer_tokenizer.pad_token_id,
            eos_token_id=infer_tokenizer.eos_token_id,
            use_cache=False,
        )

    decoded = infer_tokenizer.decode(output_ids[0], skip_special_tokens=True)

    if debug:
        print("==== RAW MODEL OUTPUT ====")
        print(decoded[:3000])
        print("==== END RAW MODEL OUTPUT ====")

    svg = extract_svg(decoded)
    svg = postprocess_svg(svg)

    if debug:
        print("==== EXTRACTED SVG ====")
        print(svg[:1000] if svg else "[EMPTY]")
        print("==== END EXTRACTED SVG ====")

    return svg

def generate_svg(prompt: str, debug: bool = False):

    svg = generate_svg_once(
        prompt,
        temperature=0.65,
        top_p=0.90,
        repetition_penalty=1.08,
        max_cap=640,
        debug=debug,
    )
    if is_valid_svg(svg):
        return svg, False

    svg = generate_svg_once(
        prompt,
        temperature=0.80,
        top_p=0.95,
        repetition_penalty=1.12,
        max_cap=2000,
        debug=debug,
    )
    if is_valid_svg(svg):
        return svg, False

    return fallback_svg(prompt), True

test_prompt = "The image features two orange squares with a microphone icon and an arrow connecting them, set against a white background."
pred_svg, used_fallback = generate_svg(test_prompt, debug=True)

print(pred_svg[:500])
print("Valid SVG:", is_valid_svg(pred_svg))
print("Used fallback:", used_fallback)
print("SVG length:", len(pred_svg))
print("Ends with </svg>:", pred_svg.strip().endswith("</svg>"))
print(pred_svg[-200:])

==((====))==  Unsloth 2026.3.18: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/qwen2.5-1.5b-instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

Both `max_new_tokens` (=640) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

==== RAW MODEL OUTPUT ====
system
You generate compact, valid SVG markup from user requests. Return only valid SVG markup. No explanations or extra text.
user
The image features two orange squares with a microphone icon and an arrow connecting them, set against a white background.
assistant
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0.0 0.0 200.0 200.0" height="200.0px" width="200.0px"><path fill="#FF6E47" fill-opacity="1.0"  filling="0" d="M58.333396911621094 108.3333969116211 L58.333396911621094 141.66659545898438 A16.666601181030273 16.666601181030273 0.0 0 1 41.666603088378906 158.33340454101562 L41.666603088378906 175.0 L158.33340454101562 175.0 L158.33340454101562 158.33340454101562 A16.666601181030273 16.666601181030273 0.0 0 1 175.0 141.66659545898438 L175.0 108.3333969116211 L133.33340454101562 108.3333969116211 L133.33340454101562 83.3333969116211 A8.333398818969727 8.333398818969727 0.0 0 1 141.66659545898438 75.0 L158.33340454101562 75.0 L158.33340454101562 83.3333969

In [22]:
full_time_min = 1.81 / 10 * len(pd.read_csv("test.csv"))
print(full_time_min)

181.0


In [23]:
TEST_PROMPTS_PATH = "test.csv"
SUBMISSION_PATH = "submission.csv"

test_df = pd.read_csv(TEST_PROMPTS_PATH)

rows = []
invalid_count = 0
t0 = time.time()
count = 0

for _, row in test_df.iterrows():
    count += 1
    print(count)
    svg, used_fallback = generate_svg(row["prompt"], debug=False)
    if used_fallback:
        invalid_count += 1
    rows.append({"id": row["id"], "svg": svg})

sub_df = pd.DataFrame(rows)
sub_df.to_csv(SUBMISSION_PATH, index=False)

elapsed_min = (time.time() - t0) / 60
print(f"Saved: {SUBMISSION_PATH}")
print(f"Rows: {len(sub_df)}")
print(f"Invalid/fallback count: {invalid_count}")
print(f"Runtime (minutes): {elapsed_min:.2f}")
sub_df.head()

Both `max_new_tokens` (=640) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=1938) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=

2


Both `max_new_tokens` (=1928) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=640) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Both `max_new_tokens` (=1897) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=640) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1000


Both `max_new_tokens` (=1915) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Saved: submission.csv
Rows: 1000
Invalid/fallback count: 197
Runtime (minutes): 671.61


,id,svg
0,fa1d8fa7-080f-4269-a9cf-a17562c9a0ca,"<svg height=""128"" viewBox=""-4 0 56 56"" width=""..."
1,6eede943219547c22ac56085027d33cc,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
2,ea045c7a247166f061ce504d9b7ccaab,"<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."
3,8fe82f3af89e487b31236ca829c3f071,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
4,600464e4d92c75338462271a09b3f176,"<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."


## Notes

- Keep a fixed seed, runtime logs, and invalid-generation counts (required by `contest_docs/05`).
- If you use Kaggle-packaged datasets (`svg-train-public`, `svg-test-public-prompts`, `svg-utils`), swap paths into the loading cells.
- For stricter alignment with Unsloth templates, copy the latest prompt-formatting snippets from the official Qwen3.5-2B notebook linked above.